# Tata Motors Stock Price Prediction using LSTM

**Project notebook:** end-to-end stock-price modelling, evaluation, visualisation and Excel export.

This notebook uses the supplied `TATAMOTORS.NS_stock_data.csv` dataset (bundled in the project ZIP). It:
1. Loads and cleans historical Tata Motors stock data.
2. Splits the data chronologically into training and test periods.
3. Fits the scaler on training data only to avoid data leakage.
4. Trains an LSTM model using the previous 60 trading days.
5. Evaluates predictions with MAE, RMSE and MAPE.
6. Displays and saves graphs.
7. Exports an Excel workbook with predictions and embedded charts.

> Educational demonstration only. The next-trading-day estimate is not investment advice or a guaranteed price.

## 1. Install dependencies

Run this cell once in a fresh environment. If TensorFlow installation fails, use Python 3.10 or 3.11 (64-bit) and create a fresh virtual environment.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow openpyxl joblib

## 2. Imports and configuration

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.chart import LineChart, Reference
from openpyxl.utils import get_column_letter

SEED = 42
LOOKBACK = 60
EPOCHS = 25
BATCH_SIZE = 32
TEST_SIZE = 0.20
TICKER = "TATAMOTORS.NS"

random.seed(SEED)
np.random.seed(SEED)

# The notebook expects the CSV in a data folder beside this notebook.
DATA_PATH = Path("data") / "TATAMOTORS.NS.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Dataset path:", DATA_PATH.resolve())
print("Output folder:", OUTPUT_DIR.resolve())

## 3. Load and clean the dataset

The supplied CSV uses `Unnamed: 0` for dates and lowercase `close` for closing prices. The code below also accepts conventional `Date` and `Close` column names.

In [ ]:
df = pd.read_csv(DATA_PATH)

date_col = next((c for c in ["Date", "date", "Unnamed: 0"] if c in df.columns), None)
close_col = next((c for c in ["Close", "close", "Adj Close", "adjclose"] if c in df.columns), None)

if date_col is None or close_col is None:
    raise ValueError("CSV must contain a date column and a closing-price column.")

df = df.rename(columns={date_col: "Date", close_col: "Close"})
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
df = (df.dropna(subset=["Date", "Close"])
        .sort_values("Date")
        .drop_duplicates(subset=["Date"], keep="last")
        .reset_index(drop=True))

if (df["Close"] <= 0).any():
    raise ValueError("Closing prices must be positive.")
if len(df) < LOOKBACK + 30:
    raise ValueError(f"At least {LOOKBACK + 30} valid records are required.")

print(f"Valid observations: {len(df):,}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
display(df.head())
display(df.tail())

## 4. Explore the historical closing price

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df["Date"], df["Close"], label="Closing price")
plt.title(f"{TICKER} Historical Closing Price")
plt.xlabel("Date")
plt.ylabel("Closing price (₹)")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Chronological train/test split and scaling

The earliest 80% of observations are used for training and the latest 20% are held out for testing. The scaler is fitted **only on the training period**. This prevents test-period information from leaking into preprocessing.

In [ ]:
close_values = df[["Close"]].to_numpy(dtype=np.float32)
split_idx = int(len(close_values) * (1 - TEST_SIZE))

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(close_values[:split_idx])
scaled_all = scaler.transform(close_values)

print("Training observations:", split_idx)
print("Testing observations:", len(close_values) - split_idx)

## 6. Create 60-day sequences

In [ ]:
def create_sequences(values, start_idx, end_idx, lookback=LOOKBACK):
    X, y = [], []
    for target_idx in range(start_idx, end_idx):
        X.append(values[target_idx - lookback:target_idx, 0])
        y.append(values[target_idx, 0])
    return np.asarray(X).reshape(-1, lookback, 1), np.asarray(y).reshape(-1, 1)

X_train, y_train = create_sequences(scaled_all, LOOKBACK, split_idx)
X_test, y_test = create_sequences(scaled_all, split_idx, len(scaled_all))

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)

## 7. Build and train the LSTM model

Architecture: three LSTM layers with 50 units each, dropout of 20%, and a single dense output. Training uses Adam and mean squared error.

In [ ]:
model = Sequential([
    Input(shape=(LOOKBACK, 1)),
    LSTM(50, return_sequences=True),
    Dropout(0.2),
    LSTM(50, return_sequences=True),
    Dropout(0.2),
    LSTM(50),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer="adam", loss="mean_squared_error")
model.summary()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

## 8. Evaluate on the unseen test period

In [ ]:
pred_scaled = model.predict(X_test, verbose=0)

actual = scaler.inverse_transform(y_test).ravel()
predicted = scaler.inverse_transform(pred_scaled).ravel()

mae = float(mean_absolute_error(actual, predicted))
rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
mape = float(np.mean(np.abs((actual - predicted) / actual)) * 100)

metrics = {"MAE": mae, "RMSE": rmse, "MAPE": mape}
metrics_df = pd.DataFrame({
    "Metric": ["MAE (₹)", "RMSE (₹)", "MAPE (%)"],
    "Value": [mae, rmse, mape]
})
display(metrics_df)

test_df = pd.DataFrame({
    "Date": df["Date"].iloc[split_idx:].reset_index(drop=True),
    "Actual": actual,
    "Predicted": predicted
})
test_df["Absolute Error"] = (test_df["Actual"] - test_df["Predicted"]).abs()
display(test_df.head())

## 9. Graph: actual vs predicted closing prices

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test_df["Date"], test_df["Actual"], label="Actual closing price")
plt.plot(test_df["Date"], test_df["Predicted"], label="Predicted closing price")
plt.title(f"{TICKER}: Actual vs Predicted Closing Price (Test Period)")
plt.xlabel("Date")
plt.ylabel("Closing price (₹)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "actual_vs_predicted.png", dpi=160)
plt.show()

## 10. Graph: training and validation loss

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training loss")
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("LSTM Training History")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_loss.png", dpi=160)
plt.show()

## 11. Estimate the next trading day's closing price

This is a model estimate based on the latest 60 available observations. It is not a guarantee and should not be used alone to make investment decisions.

In [ ]:
last_window = scaled_all[-LOOKBACK:].reshape(1, LOOKBACK, 1)
forecast_scaled = model.predict(last_window, verbose=0)
next_day_estimate = float(scaler.inverse_transform(forecast_scaled)[0, 0])

print(f"Estimated next trading-day close: ₹{next_day_estimate:.2f}")

## 12. Export results to Excel

The workbook includes:
- Summary and evaluation metrics
- Test-period actual and predicted values
- Training loss by epoch
- Historical data
- Embedded Excel line charts

Open `outputs/stock_prediction_report.xlsx` in Microsoft Excel or LibreOffice Calc.

In [ ]:
def style_sheet(ws):
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="17365D")
        cell.alignment = Alignment(horizontal="center")
    for col in ws.columns:
        letter = get_column_letter(col[0].column)
        width = min(max(max(len(str(c.value or "")) for c in col) + 2 for c in [col[0]]), 30)
        ws.column_dimensions[letter].width = max(width, 12)

wb = Workbook()
summary = wb.active
summary.title = "Summary"
summary_rows = [
    ["Tata Motors LSTM Stock Prediction", ""],
    ["Ticker", TICKER],
    ["Valid observations", len(df)],
    ["Training observations", split_idx],
    ["Testing observations", len(test_df)],
    ["Lookback window (trading days)", LOOKBACK],
    ["Next trading-day estimate (₹)", next_day_estimate],
    ["MAE (₹)", mae],
    ["RMSE (₹)", rmse],
    ["MAPE (%)", mape],
    ["Disclaimer", "Educational model output; not investment advice."]
]
for row in summary_rows:
    summary.append(row)
summary.merge_cells("A1:B1")
summary["A1"].font = Font(bold=True, size=15, color="FFFFFF")
summary["A1"].fill = PatternFill("solid", fgColor="17365D")
summary.column_dimensions["A"].width = 38
summary.column_dimensions["B"].width = 65

pred_ws = wb.create_sheet("Test Predictions")
pred_ws.append(["Date", "Actual Close (₹)", "Predicted Close (₹)", "Absolute Error (₹)"])
for row in test_df.itertuples(index=False):
    pred_ws.append([row.Date.to_pydatetime(), float(row.Actual), float(row.Predicted), float(row._3)])
for cell in pred_ws["A"][1:]:
    cell.number_format = "yyyy-mm-dd"
style_sheet(pred_ws)

history_ws = wb.create_sheet("Training History")
history_ws.append(["Epoch", "Training Loss", "Validation Loss"])
losses = history.history.get("loss", [])
val_losses = history.history.get("val_loss", [])
for i, loss in enumerate(losses, start=1):
    history_ws.append([i, float(loss), float(val_losses[i-1]) if i-1 < len(val_losses) else None])
style_sheet(history_ws)

raw_ws = wb.create_sheet("Historical Data")
raw_ws.append(list(df.columns))
for row in df.itertuples(index=False, name=None):
    raw_ws.append([v.to_pydatetime() if isinstance(v, pd.Timestamp) else v for v in row])
style_sheet(raw_ws)
if "Date" in df.columns:
    for cell in raw_ws["A"][1:]:
        cell.number_format = "yyyy-mm-dd"

charts_ws = wb.create_sheet("Charts")
price_chart = LineChart()
price_chart.title = "Actual vs Predicted Closing Price"
price_chart.y_axis.title = "Closing price (₹)"
price_chart.x_axis.title = "Test date"
price_chart.height, price_chart.width = 10, 22
price_data = Reference(pred_ws, min_col=2, max_col=3, min_row=1, max_row=pred_ws.max_row)
price_dates = Reference(pred_ws, min_col=1, min_row=2, max_row=pred_ws.max_row)
price_chart.add_data(price_data, titles_from_data=True)
price_chart.set_categories(price_dates)
charts_ws.add_chart(price_chart, "A1")

loss_chart = LineChart()
loss_chart.title = "Training Loss by Epoch"
loss_chart.y_axis.title = "Mean squared error"
loss_chart.x_axis.title = "Epoch"
loss_chart.height, loss_chart.width = 10, 18
loss_data = Reference(history_ws, min_col=2, max_col=3, min_row=1, max_row=history_ws.max_row)
loss_chart.add_data(loss_data, titles_from_data=True)
loss_chart.set_categories(Reference(history_ws, min_col=1, min_row=2, max_row=history_ws.max_row))
charts_ws.add_chart(loss_chart, "A22")

excel_file = OUTPUT_DIR / "stock_prediction_report.xlsx"
wb.save(excel_file)
print("Excel report saved to:", excel_file.resolve())

## 13. Save the trained model and prediction outputs

In [ ]:
import joblib

model.save(OUTPUT_DIR / "lstm_stock_model.keras")
joblib.dump(scaler, OUTPUT_DIR / "price_scaler.pkl")
test_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

with open(OUTPUT_DIR / "model_metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "ticker": TICKER,
        "metrics": metrics,
        "next_trading_day_estimate": next_day_estimate
    }, f, indent=2)

print("Saved files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)

## 14. Conclusion and limitations

- The LSTM learns patterns from the historical closing-price sequence.
- Test metrics describe performance on the chronological holdout period, not future certainty.
- This model uses closing price alone; it does not incorporate news, market indices, fundamentals, corporate actions or other external factors.
- Historical stock prices and adjusted-price conventions may change across data providers. Review the source and intended price field before interpreting results.
- This notebook is an educational project, not financial advice.